# LP3 Practical 11 – Diabetes Prediction using KNN
**Aim:** Implement K-Nearest Neighbors on `diabetes.csv` and calculate confusion matrix, accuracy, error rate, precision and recall.

**Target:** `Outcome` (0 = no diabetes, 1 = diabetes).

## Cell 1 – Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    classification_report
)


## Cell 2 – Load and inspect dataset

In [ ]:
from pathlib import Path

def get_dataset_path(filename):
    candidates = [
        Path.cwd() / "Datasets" / filename,
        Path.cwd().parent / "Datasets" / filename
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find {filename}. Keep the Datasets folder beside the Notebooks folder."
    )

DATA_PATH = get_dataset_path("diabetes.csv")
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())
print(df.info())
print(df["Outcome"].value_counts())


## Cell 3 – Preprocess data

In [ ]:
# In this dataset, zero is not a realistic measurement for these columns.
# Replace those zero values with NaN, then fill them with the column median.
zero_is_missing = [
    "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"
]

for column in zero_is_missing:
    df[column] = df[column].replace(0, np.nan)
    df[column] = df[column].fillna(df[column].median())

print("Missing values after preprocessing:")
print(df.isnull().sum())


## Cell 4 – Explore outliers and correlation

In [ ]:
plt.figure(figsize=(10, 5))
df.drop(columns=["Outcome"]).boxplot(rot=45)
plt.title("Box Plot of Diabetes Features")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
corr = df.corr(numeric_only=True)
plt.imshow(corr, cmap="coolwarm", aspect="auto")
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


## Cell 5 – Separate features and target; split data

In [ ]:
X = df.drop(columns=["Outcome"])
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)


## Cell 6 – Feature scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## Cell 7 – Train KNN model

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)

predictions = knn_model.predict(X_test_scaled)
print("KNN model trained.")


## Cell 8 – Evaluate model

In [ ]:
cm = confusion_matrix(y_test, predictions)
accuracy = accuracy_score(y_test, predictions)
error_rate = 1 - accuracy
precision = precision_score(y_test, predictions, zero_division=0)
recall = recall_score(y_test, predictions, zero_division=0)

print("Confusion Matrix:")
print(cm)
print("Accuracy:", accuracy)
print("Error Rate:", error_rate)
print("Precision:", precision)
print("Recall:", recall)

print("\nClassification Report:")
print(classification_report(y_test, predictions, zero_division=0))


## Sample output / observation
- `Outcome = 0`: patient is classified as not having diabetes.
- `Outcome = 1`: patient is classified as having diabetes.
- Accuracy = correct predictions / total predictions.
- Error rate = 1 − accuracy.
- Precision and recall are calculated for the positive class (Outcome = 1).